# SCE Net + Deep SVDD для one-class compatibility (anomaly detection)

> Цель: переобучить пайплайн совместимости так, чтобы в `train/val` использовались **только позитивные пары** (`y=1`), а на тесте (good/bad) выбирался порог по anomaly score.

Этот ноутбук — подробный практический план, как адаптировать архитектуру SCE Net под one-class постановку в стиле **Deep SVDD**.

## 1. Почему это может решить вашу текущую проблему

В текущей бинарной постановке качество сильно зависит от того, как вы синтезируете негативы. Если негативы «переусложнены» или распределены не как реальные `bad` пары стилиста, модель начинает делать ложные срабатывания даже на базовых комбинациях (например, белая футболка + джинсы).

Идея one-class: учим компактное представление **только класса `good`**, а всё, что далеко от learned-good manifold, считаем аномалией.

## 2. Связка SCE Net и Deep SVDD: ключевая адаптация

Сохраняем pair-conditioned representation SCE Net (`condition_masks` + `weight_branch`), а triplet-loss заменяем на objective Deep SVDD:

`L_svdd = mean(||z_i - c||^2) + reg`.

Где `c` — центр класса good, `z_i` — pair embedding. На инференсе anomaly score: `s(x)=||z-c||^2`.

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 3. Данные

- `train_pairs`: только `y=1`
- `val_pairs`: только `y=1`
- `test_pairs`: `y in {0,1}` от стилиста для оценки и порога

In [ ]:
train_pairs = pd.read_csv('data/train_pairs_positive.csv')
val_pairs   = pd.read_csv('data/val_pairs_positive.csv')
test_pairs  = pd.read_csv('data/test_pairs_labeled.csv')

assert set(train_pairs['y'].unique()) <= {1}
assert set(val_pairs['y'].unique()) <= {1}
assert set(test_pairs['y'].unique()) <= {0,1}

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs_df, sku_to_feat):
        self.df = pairs_df.reset_index(drop=True)
        self.sku_to_feat = sku_to_feat

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'x1': torch.tensor(self.sku_to_feat[row['sku1']], dtype=torch.float32),
            'x2': torch.tensor(self.sku_to_feat[row['sku2']], dtype=torch.float32),
            'y': torch.tensor(int(row['y']), dtype=torch.long),
        }

In [ ]:
class SCENetOneClass(nn.Module):
    def __init__(self, d_in=512, d_proj=256, num_masks=8, hidden=512):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, d_proj)
        )
        self.condition_masks = nn.Parameter(torch.ones(num_masks, d_proj))
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * d_proj, hidden), nn.ReLU(), nn.Linear(hidden, num_masks)
        )

    def forward(self, x1, x2):
        v1 = self.backbone(x1)
        v2 = self.backbone(x2)
        w = F.softmax(self.weight_branch(torch.cat([v1, v2], dim=-1)), dim=-1)

        m = self.condition_masks.unsqueeze(0)
        e1 = v1.unsqueeze(1) * m
        e2 = v2.unsqueeze(1) * m
        z1 = torch.einsum('bm,bmd->bd', w, e1)
        z2 = torch.einsum('bm,bmd->bd', w, e2)
        z = torch.abs(z1 - z2)
        return {'z': z, 'w': w, 'z1': z1, 'z2': z2}

In [ ]:
@torch.no_grad()
def init_center_c(model, loader, device='cpu', eps=1e-4):
    model.eval()
    zs = []
    for batch in tqdm(loader, desc='init_center'):
        out = model(batch['x1'].to(device), batch['x2'].to(device))
        zs.append(out['z'])
    c = torch.cat(zs, dim=0).mean(dim=0)
    c[(c.abs() < eps) & (c < 0)] = -eps
    c[(c.abs() < eps) & (c > 0)] = eps
    return c

def deep_svdd_loss(z, c):
    return torch.mean(torch.sum((z - c) ** 2, dim=1))

In [ ]:
@dataclass
class TrainConfig:
    lr: float = 1e-4
    weight_decay: float = 1e-6
    lambda_mask_l1: float = 1e-5
    epochs: int = 20


def train_one_class(model, train_loader, val_loader, cfg: TrainConfig, device='cpu'):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    c = init_center_c(model, train_loader, device=device)

    best_val, best_state = float('inf'), None
    for epoch in range(1, cfg.epochs + 1):
        model.train(); tr=[]
        for b in tqdm(train_loader, desc=f'train ep{epoch}'):
            out = model(b['x1'].to(device), b['x2'].to(device))
            loss = deep_svdd_loss(out['z'], c) + cfg.lambda_mask_l1 * model.condition_masks.abs().mean()
            opt.zero_grad(); loss.backward(); opt.step(); tr.append(loss.item())

        model.eval(); va=[]
        with torch.no_grad():
            for b in tqdm(val_loader, desc=f'val ep{epoch}'):
                out = model(b['x1'].to(device), b['x2'].to(device))
                l = deep_svdd_loss(out['z'], c) + cfg.lambda_mask_l1 * model.condition_masks.abs().mean()
                va.append(l.item())
        trm, vam = float(np.mean(tr)), float(np.mean(va))
        print(f'Epoch {epoch:03d} | train={trm:.6f} | val={vam:.6f}')
        if vam < best_val:
            best_val = vam
            best_state = {'model': {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}, 'c': c.detach().cpu().clone()}

    model.load_state_dict(best_state['model'])
    return model, best_state['c'].to(device)

In [ ]:
@torch.no_grad()
def predict_scores(model, c, loader, device='cpu'):
    model.eval(); scores=[]; labels=[]
    for b in tqdm(loader, desc='predict'):
        out = model(b['x1'].to(device), b['x2'].to(device))
        s = torch.sum((out['z'] - c) ** 2, dim=1).cpu().numpy()
        scores.append(s); labels.append(b['y'].cpu().numpy())
    return np.concatenate(scores), np.concatenate(labels)


def find_best_threshold(y_true, scores):
    y_anom = (y_true == 0).astype(int)
    p, r, t = precision_recall_curve(y_anom, scores)
    f1 = 2 * p * r / (p + r + 1e-12)
    i = np.argmax(f1[:-1])
    return t[i], {'best_f1': float(f1[:-1][i]), 'precision': float(p[:-1][i]), 'recall': float(r[:-1][i])}

## 8. Практические рекомендации

1. Делайте две группы LR: меньше для backbone, больше для masks/weight_branch.
2. Добавьте симметричную аугментацию пар (swap sku1/sku2).
3. Следите за false negatives на простых good луках; если их много, ослабьте компактность (L1/WD), увеличьте diversity positives и/или число masks.
4. Порог выбирайте на отложенной выборке с реальными bad от стилиста.
5. Логируйте `w` по masks для explainability провалов.